# 📐 CMA-ES (Covariance Matrix Adaptation Evolution Strategy)

## Teori

CMA-ES, 2001 yılında Nikolaus Hansen tarafından geliştirilen, **yüksek boyutlu sürekli optimizasyon** için tasarlanmış evrimsel bir stratejidir.

### Ana Fikir
Gradyan hesaplamadan, **kovaryans matrisini adapte ederek** arama dağılımının şeklini öğrenir. Sanki algoritma kendi içinde "hangi yönlerde arama yapmalıyım?" sorusunu yanıtlar.

### Kovaryans Matrisi Ne İşe Yarar?
```
Kovaryans matrisi C, arama dağılımının:
  ├── Şeklini (hangi yönde uzanıyor?)
  ├── Yönelimini (kaç derecelik açıyla?)
  └── Büyüklüğünü (adım boyutu σ) tanımlar
```

### Matematiksel Temel
Her nesilde yeni adaylar şu dağılımdan örneklenir:

$$\mathbf{x}_k \sim \mathcal{N}(\mathbf{m}, \sigma^2 \mathbf{C})$$

- $\mathbf{m}$: Ortalama vektör (mevcut en iyi tahmin)
- $\sigma$: Adım boyutu (arama yarıçapı)  
- $\mathbf{C}$: Kovaryans matrisi (arama şekli)

### Güncelleme Adımları
| Güncelleme | Ne Yapar? | Neden? |
|---|---|---|
| **Mean update** | Ortalamayı en iyi adaylara doğru kaydır | Ümit verici bölgeye yaklaş |
| **Step-size control** | σ'yı büyüt/küçült | Arama hassasiyetini ayarla |
| **Covariance update** | C'yi başarılı yönlerle güncelle | Doğru yönde arama yap |

In [ ]:
# ============================================================
# KÜTÜPHANE İMPORTLARI
# ============================================================
import numpy as np                    # Temel sayısal işlemler
import matplotlib.pyplot as plt       # Görselleştirme
from matplotlib.patches import Ellipse  # Kovaryans ellipsi çizmek için
import matplotlib.transforms as transforms
import warnings
warnings.filterwarnings('ignore')

# Tekrar üretilebilirlik
np.random.seed(42)
print("✅ Kütüphaneler yüklendi.")

## 1. Test Fonksiyonları

In [ ]:
# ============================================================
# TEST FONKSİYONLARI
# ============================================================

def rosenbrock(x):
    """
    Rosenbrock (Muz) Fonksiyonu — CMA-ES'in klasik test problemi.
    
    Formül: f(x) = Σ [100(x_{i+1} - x_i²)² + (1 - x_i)²]
    Minimum: f(1,1,...,1) = 0
    
    Özellik: Derin, dar, kıvrımlı vadi → gradyan yöntemleri takılır,
    CMA-ES kovaryansı adapte ederek bu vadi boyunca ilerler.
    """
    return sum(
        100.0 * (x[i+1] - x[i]**2)**2 + (1 - x[i])**2
        for i in range(len(x) - 1)
    )


def rastrigin(x):
    """Rastrigin fonksiyonu — çok-modlu benchmark."""
    A = 10
    return A * len(x) + np.sum(x**2 - A * np.cos(2 * np.pi * x))


def ellipsoid(x):
    """
    Ellipsoid (Elipsoid) Fonksiyonu.
    
    Formül: f(x) = Σ [10^(6i/(n-1)) · x_i²]
    Minimum: f(0,...,0) = 0
    
    Özellik: Her eksende farklı ölçek → kovaryans adaptasyonunu test eder.
    Gradyan yöntemleri için koşul sayısı 10^6 olabilir!
    """
    n = len(x)
    exponents = np.array([6.0 * i / (n - 1) for i in range(n)])
    return np.sum((10 ** exponents) * x**2)


# Fonksiyonları karşılaştır
test_point = np.array([1.0, 1.0])
optimal    = np.array([0.0, 0.0])

print("Test noktası (1,1):")
print(f"  Rosenbrock : {rosenbrock(test_point):.4f}  (minimum: 0 @ (1,1))")
print(f"  Rastrigin  : {rastrigin(test_point):.4f}   (minimum: 0 @ (0,0))")
print(f"  Ellipsoid  : {ellipsoid(test_point):.4f}  (minimum: 0 @ (0,0))")

## 2. CMA-ES Implementasyonu

CMA-ES'in tam implementasyonu. Her güncelleme adımı detaylıca açıklanmıştır.

In [ ]:
# ============================================================
# CMA-ES SINIFI — TAM İMPLEMENTASYON
# ============================================================

class CMAES:
    """
    CMA-ES (Covariance Matrix Adaptation Evolution Strategy)
    
    Referans: Hansen, N. (2016). The CMA Evolution Strategy: A Tutorial.
    arXiv:1604.00772
    """
    
    def __init__(self, n_dims, x0=None, sigma0=0.5):
        """
        CMA-ES başlangıç parametrelerini ve iç durumunu başlat.
        
        Parametreler:
            n_dims (int)       : Problem boyutu (n)
            x0     (np.ndarray): Başlangıç ortalama noktası
            sigma0 (float)     : Başlangıç adım boyutu (σ₀)
        """
        self.n = n_dims    # Kısa isim
        n = n_dims
        
        # ── BAŞLANGIÇ NOKTASI ────────────────────────────────
        # m: Arama dağılımının merkezi (mevcut en iyi tahmin)
        self.mean = x0 if x0 is not None else np.random.randn(n)
        
        # σ: Adım boyutu — arama yarıçapını kontrol eder
        self.sigma = sigma0
        
        # ── STRATEJİ PARAMETRELERİ ───────────────────────────
        # Hansen'ın önerdiği varsayılan değerler:
        
        # λ: Her nesilde üretilen aday sayısı
        self.lam = 4 + int(3 * np.log(n))    # Formül: 4 + 3·ln(n)
        
        # μ: En iyi kaç aday kullanılacak (üst %50)
        self.mu = self.lam // 2
        
        # ── AĞIRLIKLAR: İyi adaylara daha fazla ağırlık ver ──
        # log(μ+0.5) - log(i) formülü ile hesaplanır
        weights_raw = np.log(self.mu + 0.5) - np.log(np.arange(1, self.mu + 1))
        self.weights = weights_raw / weights_raw.sum()    # Normalize et (toplam=1)
        
        # μ_eff: Etkin seçim sayısı (variance effective selection mass)
        # Ağırlıklı ortalama için sapma ölçüsü
        self.mu_eff = 1.0 / np.sum(self.weights**2)
        
        # ── ADIM BOYUTU KONTROL PARAMETRELERİ ───────────────
        # cs: Adım boyutu kümülatif yolu için öğrenme oranı
        self.cs = (self.mu_eff + 2) / (n + self.mu_eff + 5)
        
        # ds: Adım boyutu sönümleme faktörü
        self.ds = 1 + 2 * max(0, np.sqrt((self.mu_eff - 1) / (n + 1)) - 1) + self.cs
        
        # χ_n: n-boyutlu standart normal'in beklenen normu
        # Yaklaşık formül: √n · (1 - 1/(4n) + 1/(21n²))
        self.chiN = np.sqrt(n) * (1 - 1 / (4 * n) + 1 / (21 * n**2))
        
        # ── KOVARYANS GÜNCELLEMESİ PARAMETRELERİ ────────────
        # cc: Kovaryans kümülatif yolu için öğrenme oranı
        self.cc = (4 + self.mu_eff / n) / (n + 4 + 2 * self.mu_eff / n)
        
        # c1: Rank-1 güncelleme öğrenme oranı (tek en iyi yön)
        self.c1 = 2 / ((n + 1.3)**2 + self.mu_eff)
        
        # cmu: Rank-μ güncelleme öğrenme oranı (tüm seçilen adaylar)
        self.cmu = min(
            1 - self.c1,
            2 * (self.mu_eff - 2 + 1/self.mu_eff) / ((n + 2)**2 + self.mu_eff)
        )
        
        # ── İÇ DURUM DEĞİŞKENLERİ ────────────────────────────
        # ps: Adım boyutu için evrim yolu (kümülatif hareket)
        self.ps = np.zeros(n)
        
        # pc: Kovaryans için evrim yolu
        self.pc = np.zeros(n)
        
        # C: Kovaryans matrisi — başlangıçta birim matris (küresel arama)
        self.C = np.eye(n)
        
        # Nesil sayacı
        self.generation = 0
        
        print(f"CMA-ES başlatıldı: n={n}, λ={self.lam}, μ={self.mu}")
        print(f"  Adım boyutu öğrenme oranı  cs={self.cs:.4f}")
        print(f"  Kovaryans öğrenme oranları: cc={self.cc:.4f}, c1={self.c1:.4f}, cmu={self.cmu:.4f}")
    
    
    def ask(self):
        """
        Yeni λ aday çözüm üret (örnekleme adımı).
        
        Çok değişkenli normal dağılımdan örnekleme:
        x_k = m + σ · C^(1/2) · z_k    burada z_k ~ N(0,I)
        
        Döndürür:
            candidates (list): λ adet aday çözüm
            z_vectors  (list): Ham normal vektörler (güncellemede kullanılır)
        """
        # Kovaryans matrisinin Cholesky ayrışımı: C = L·L^T
        # Bu sayede C^(1/2) hesaplanır
        try:
            L = np.linalg.cholesky(self.C)    # Alt üçgen matris
        except np.linalg.LinAlgError:
            # Sayısal instabilite → C'yi pozitif tanımlı yap
            self.C = (self.C + self.C.T) / 2 + 1e-10 * np.eye(self.n)
            L = np.linalg.cholesky(self.C)
        
        candidates = []
        z_vectors  = []
        
        for _ in range(self.lam):
            # z: Standart normal'den örnekle N(0, I)
            z = np.random.randn(self.n)
            # x = m + σ · L · z  →  N(m, σ²C) dağılımından aday
            x = self.mean + self.sigma * L @ z
            candidates.append(x)
            z_vectors.append(z)
        
        return candidates, z_vectors
    
    
    def tell(self, candidates, fitness_values, z_vectors):
        """
        Fitness değerlerine göre dağılımı güncelle.
        
        Bu adımda üç güncelleme yapılır:
          1. Ortalama (mean) güncelleme
          2. Adım boyutu (sigma) güncelleme — Cumulative Step Adaptation
          3. Kovaryans matrisi (C) güncelleme — Rank-1 + Rank-μ
        
        Parametreler:
            candidates    (list): ask() ile üretilen adaylar
            fitness_values(list): Her adayın fitness değeri
            z_vectors     (list): ask() deki ham normal vektörler
        """
        n = self.n
        
        # ── En iyi μ adayı sırala (düşük fitness = iyi) ─────
        sorted_idx = np.argsort(fitness_values)
        best_candidates = [candidates[i] for i in sorted_idx[:self.mu]]
        best_z          = [z_vectors[i]  for i in sorted_idx[:self.mu]]
        
        # ── ADIM 1: ORTALAMA GÜNCELLEME ─────────────────────
        # Yeni ortalama = en iyi μ adayın ağırlıklı ortalaması
        # Neden? En iyi çözümlerin bulunduğu bölgeye doğru kaydır
        old_mean = self.mean.copy()
        self.mean = np.sum(
            [w * x for w, x in zip(self.weights, best_candidates)],
            axis=0
        )
        
        # Ortalama hareketi (σ ile normalize edilmiş)
        mean_shift = (self.mean - old_mean) / self.sigma
        
        # ── ADIM 2: ADIM BOYUTU EVRİM YOLU (ps) ─────────────
        # ps: Tarihsel hareket yönünü biriktirir ("momentum")
        # C^(-1/2) ile beyazlatılmış adım — yönden bağımsız ölçüm
        try:
            C_invsqrt = np.linalg.inv(np.linalg.cholesky(self.C)).T
        except:
            C_invsqrt = np.eye(n)
        
        # ps güncelleme: Üstel hareketli ortalama
        # (1 - cs): Eski yolu unut    cs: Yeni bilgiyi ekle
        self.ps = (1 - self.cs) * self.ps + \
                  np.sqrt(self.cs * (2 - self.cs) * self.mu_eff) * (C_invsqrt @ mean_shift)
        
        # ── ADIM 3: ADIM BOYUTU GÜNCELLEME ──────────────────
        # Fikir: ps uzunluğu chiN'e eşitse σ doğru;
        #        ps uzunluğu > chiN ise σ çok küçük → büyüt
        #        ps uzunluğu < chiN ise σ çok büyük → küçült
        sigma_update = (self.cs / self.ds) * (np.linalg.norm(self.ps) / self.chiN - 1)
        self.sigma *= np.exp(sigma_update)
        self.sigma  = np.clip(self.sigma, 1e-10, 1e10)  # Sayısal stabilite
        
        # ── ADIM 4: KOVARYANs EVRİM YOLU (pc) ──────────────
        # pc: Başarılı hareket yönlerini biriktirir
        # h_sigma: ps çok büyükse rank-1 güncellemeyi durdur (stalled)
        h_sigma_lhs = np.linalg.norm(self.ps) / np.sqrt(1 - (1 - self.cs)**(2*(self.generation+1)))
        h_sigma = 1 if h_sigma_lhs < (1.4 + 2/(n+1)) * self.chiN else 0
        
        self.pc = (1 - self.cc) * self.pc + \
                  h_sigma * np.sqrt(self.cc * (2 - self.cc) * self.mu_eff) * mean_shift
        
        # ── ADIM 5: KOVARYANs GÜNCELLEMESİ ─────────────────
        # Rank-1 güncelleme: pc vektörünün dış çarpımı
        # Bu güncelleme tek başarılı yönü öğrenir ("memory")
        rank1 = np.outer(self.pc, self.pc)  # pc · pc^T matris
        
        # Rank-μ güncelleme: En iyi μ adayın ağırlıklı dış çarpımı
        # Bu güncelleme birden fazla yönü eş zamanlı öğrenir
        rank_mu = np.zeros((n, n))
        for w, z in zip(self.weights, best_z):
            rank_mu += w * np.outer(z, z)    # z · z^T ağırlıklı toplam
        
        # C güncelleme: Eski C'yi unut + yeni yönleri ekle
        delta_h = (1 - h_sigma) * self.cc * (2 - self.cc)
        self.C = (
            (1 + self.c1 * delta_h - self.c1 - self.cmu) * self.C +
            self.c1  * rank1    +    # Rank-1: Tek yönlü hafıza
            self.cmu * rank_mu       # Rank-μ: Çoklu yön öğrenmesi
        )
        
        # Sayısal simetri koru (yuvarlama hataları giderilir)
        self.C = (self.C + self.C.T) / 2
        
        self.generation += 1


print("✅ CMA-ES sınıfı tanımlandı.")

## 3. Optimizasyon Döngüsü

In [ ]:
# ============================================================
# CMA-ES OPTİMİZASYON DÖNGÜSÜ
# ============================================================

def run_cmaes(
    fitness_fn,          # Optimize edilecek fonksiyon
    n_dims     = 10,     # Problem boyutu
    x0         = None,   # Başlangıç noktası (None → rastgele)
    sigma0     = 1.0,    # Başlangıç adım boyutu
    max_gens   = 500,    # Maksimum nesil
    tol        = 1e-10,  # Yakınsama toleransı
    verbose    = True    # İlerlemeyi yazdır
):
    """
    CMA-ES optimizasyon döngüsünü çalıştırır.
    
    Döndürür:
        best_solution (np.ndarray): En iyi çözüm
        best_fitness  (float)     : En iyi fitness
        history       (dict)      : İlerleme kaydı
    """
    # Başlangıç noktası
    if x0 is None:
        x0 = np.random.uniform(-3, 3, n_dims)
    
    # CMA-ES nesnesini oluştur
    cma = CMAES(n_dims=n_dims, x0=x0, sigma0=sigma0)
    
    # İstatistik kaydı
    history = {
        'best'      : [],    # En iyi fitness
        'sigma'     : [],    # Adım boyutu
        'cond_C'    : [],    # C'nin koşul sayısı (kondisyon)
        'mean_norm' : [],    # Ortalama vektörün normu
    }
    
    best_overall     = np.inf
    best_overall_sol = x0.copy()
    
    for gen in range(max_gens):
        # ── 1. ÖRNEKLEME: λ aday üret ──────────────────────
        candidates, z_vectors = cma.ask()
        
        # ── 2. DEĞERLENDİRME: Her adayın fitness'ını hesapla
        fitness_values = [fitness_fn(x) for x in candidates]
        
        # ── 3. GÜNCELLEME: Dağılımı iyileştir ──────────────
        cma.tell(candidates, fitness_values, z_vectors)
        
        # ── İSTATİSTİKLER ───────────────────────────────────
        gen_best = min(fitness_values)
        if gen_best < best_overall:
            best_overall     = gen_best
            best_overall_sol = candidates[np.argmin(fitness_values)].copy()
        
        # Kovaryans matrisinin koşul sayısı (eigenvalue oranı)
        eigenvalues = np.linalg.eigvalsh(cma.C)    # Simetrik matris için
        cond_C = np.max(eigenvalues) / np.max([np.min(eigenvalues), 1e-20])
        
        history['best'].append(best_overall)
        history['sigma'].append(cma.sigma)
        history['cond_C'].append(cond_C)
        history['mean_norm'].append(np.linalg.norm(cma.mean))
        
        # Loglama
        if verbose and (gen < 5 or (gen+1) % 100 == 0):
            print(f"Nesil {gen+1:4d} | f={best_overall:.2e} | "
                  f"σ={cma.sigma:.4f} | cond(C)={cond_C:.1f}")
        
        # ── DURDURMA KRİTERLERİ ─────────────────────────────
        # 1. Yeterince iyi çözüm bulunduysa
        if best_overall < tol:
            print(f"\n✅ Yakınsama! Nesil {gen+1}: f = {best_overall:.2e} < tol={tol}")
            break
        
        # 2. Adım boyutu çok küçüldüyse (arama durdu)
        if cma.sigma < 1e-12:
            print(f"\n⚠️  Adım boyutu çok küçük: σ = {cma.sigma:.2e}")
            break
    
    return best_overall_sol, best_overall, history


# ── ÇALIŞTIR: Rosenbrock (kıvrımlı vadi) ────────────────────
print("\n" + "="*55)
print("Rosenbrock optimizasyonu başlatılıyor (10 boyut)...")
print("="*55)

best_sol, best_fit, hist = run_cmaes(
    fitness_fn = rosenbrock,
    n_dims     = 10,
    sigma0     = 1.0,
    max_gens   = 500,
    tol        = 1e-8
)

print(f"\n{'='*55}")
print(f"Sonuç: f = {best_fit:.2e}")
print(f"Teorik minimum: 0 @ (1,1,...,1)")
print(f"Ortalama hata : {np.mean(np.abs(best_sol - 1.0)):.2e}")

In [ ]:
# ============================================================
# SONUÇ GÖRSELLEŞTİRME
# ============================================================

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('CMA-ES — Optimizasyon Analizi', fontsize=15, fontweight='bold')

gens = np.arange(1, len(hist['best']) + 1)

# ── Graf 1: Konverjans ────────────────────────────────────
axes[0,0].semilogy(gens, hist['best'], 'b-', linewidth=2)
axes[0,0].set_xlabel('Nesil'); axes[0,0].set_ylabel('En İyi f(x) [log ölçek]')
axes[0,0].set_title('Konverjans Eğrisi')
axes[0,0].grid(True, alpha=0.3, which='both')
axes[0,0].axhline(y=1e-8, color='red', linestyle='--', alpha=0.7, label='Tolerans (1e-8)')
axes[0,0].legend()

# ── Graf 2: Adım Boyutu Evrimi ────────────────────────────
axes[0,1].semilogy(gens, hist['sigma'], 'g-', linewidth=2)
axes[0,1].set_xlabel('Nesil'); axes[0,1].set_ylabel('σ (Adım Boyutu) [log ölçek]')
axes[0,1].set_title('Adım Boyutu Adaptasyonu')
axes[0,1].grid(True, alpha=0.3, which='both')
# σ'nın küçülmesi yakınsama, büyümesi keşif anlamına gelir
axes[0,1].annotate('Yakınsama:\nσ küçülüyor', 
    xy=(gens[-1]*0.7, hist['sigma'][int(len(hist['sigma'])*0.7)]),
    fontsize=9, color='darkgreen')

# ── Graf 3: Kovaryans Koşul Sayısı ────────────────────────
axes[1,0].semilogy(gens, hist['cond_C'], 'r-', linewidth=2)
axes[1,0].set_xlabel('Nesil'); axes[1,0].set_ylabel('cond(C) [log ölçek]')
axes[1,0].set_title('Kovaryans Matrisi Koşul Sayısı')
axes[1,0].grid(True, alpha=0.3, which='both')
# Yüksek koşul sayısı → C çok uzamış (uzun vadi keşfi)
# Çok yüksek (>10^14) → sayısal instabilite

# ── Graf 4: Ortalama Normu ────────────────────────────────
axes[1,1].semilogy(gens, hist['mean_norm'], 'm-', linewidth=2)
axes[1,1].set_xlabel('Nesil'); axes[1,1].set_ylabel('||m|| (Ortalama Normu)')
axes[1,1].set_title('Ortalama Vektörün Normu')
axes[1,1].grid(True, alpha=0.3, which='both')
# Rosenbrock: minimum @ (1,...,1) → ||m|| → √n ≈ 3.16
axes[1,1].axhline(y=np.sqrt(10), color='red', linestyle='--', 
                   label=f'Teorik: √10 ≈ {np.sqrt(10):.2f}')
axes[1,1].legend()

plt.tight_layout()
plt.savefig('cmaes_sonuclar.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Kovaryans Matrisi Adaptasyonu — Görselleştirme

2D örnekte kovaryans elipsinin nasıl şekillendiğini görelim.

In [ ]:
# ============================================================
# 2D KOVARYANS ELİPSİ ANİMASYONU
# ============================================================

def plot_covariance_ellipse(ax, mean, cov, sigma, n_std=2.0, **kwargs):
    """
    Kovaryans matrisini 2D elips olarak çiz.
    
    Parametreler:
        mean  (np.ndarray): Elips merkezi [x, y]
        cov   (np.ndarray): 2x2 kovaryans matrisi
        sigma (float)     : Adım boyutu (elipsi ölçekler)
        n_std (float)     : Kaç standart sapma çizilsin?
    """
    # Özdeger ayrışımı: C = V · Λ · V^T
    # eigenvalues → elips boyları, eigenvectors → elips yönleri
    eigenvalues, eigenvectors = np.linalg.eigh(sigma**2 * cov)
    
    # Elips yarı eksenleri: √eigenvalue ölçeklendirilmiş
    width  = 2 * n_std * np.sqrt(max(eigenvalues[0], 0))
    height = 2 * n_std * np.sqrt(max(eigenvalues[1], 0))
    
    # Elipsin dönüş açısı (birinci özvektörün açısı)
    angle_rad = np.arctan2(eigenvectors[1, 0], eigenvectors[0, 0])
    angle_deg = np.degrees(angle_rad)
    
    ellipse = Ellipse(
        xy=(mean[0], mean[1]),
        width=width, height=height,
        angle=angle_deg,
        **kwargs
    )
    ax.add_patch(ellipse)


# ── 2D CMA-ES İzleme ─────────────────────────────────────
# Ellipsoid fonksiyonu üzerinde kovaryans adaptasyonunu gözlemle
def ellipsoid_2d(x):
    """2D ellipsoid: farklı eksen ölçekleri kovaryansı zorlar."""
    return 100 * x[0]**2 + x[1]**2    # x ekseni 100x daha hassas


np.random.seed(42)
x0_2d = np.array([3.0, 3.0])    # Başlangıç noktası

cma_2d = CMAES(n_dims=2, x0=x0_2d, sigma0=1.0)

# Snapshot'lar için kaydedelim
snapshots = []    # (mean, C, sigma) üçlüleri
best_fits_2d = []

for gen in range(200):
    candidates, z_vectors = cma_2d.ask()
    fitness_values = [ellipsoid_2d(x) for x in candidates]
    cma_2d.tell(candidates, fitness_values, z_vectors)
    
    # Her 20 neslide snapshot al
    if gen % 20 == 0:
        snapshots.append((
            cma_2d.mean.copy(),
            cma_2d.C.copy(),
            cma_2d.sigma
        ))
    best_fits_2d.append(min(fitness_values))


# Görselleştirme
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('CMA-ES Kovaryans Adaptasyonu — Ellipsoid 2D', fontsize=14, fontweight='bold')

# ── Sol: Elips evrimi ─────────────────────────────────────
x1 = np.linspace(-4, 4, 200); x2 = np.linspace(-4, 4, 200)
X1, X2 = np.meshgrid(x1, x2)
Z2d = 100 * X1**2 + X2**2    # Ellipsoid konturu

axes[0].contourf(X1, X2, Z2d, levels=30, cmap='Blues', alpha=0.4)
axes[0].contour(X1, X2, Z2d, levels=30, colors='gray', alpha=0.3, linewidths=0.5)

# Farklı nesillerdeki kovaryans elipslerini çiz
colors_snap = plt.cm.Reds(np.linspace(0.3, 1.0, len(snapshots)))
for i, (mean_s, C_s, sigma_s) in enumerate(snapshots):
    plot_covariance_ellipse(
        axes[0], mean_s, C_s, sigma_s,
        fill=False, edgecolor=colors_snap[i],
        linewidth=2, label=f'Nesil {i*20}' if i % 2 == 0 else None
    )
    axes[0].scatter(*mean_s, color=colors_snap[i], s=30, zorder=5)

axes[0].scatter(0, 0, color='red', s=200, marker='*', zorder=10, label='Minimum (0,0)')
axes[0].set_xlim(-4.5, 4.5); axes[0].set_ylim(-4.5, 4.5)
axes[0].set_xlabel('x₁'); axes[0].set_ylabel('x₂')
axes[0].set_title('Kovaryans Elipsinin Adaptasyonu\n(Kırmızı→Açık kırmızı: İlk→Son)')
axes[0].legend(fontsize=8, loc='upper right'); axes[0].set_aspect('equal')

# ── Sağ: Konverjans ───────────────────────────────────────
axes[1].semilogy(range(1, 201), best_fits_2d, 'b-', linewidth=2)
axes[1].set_xlabel('Nesil'); axes[1].set_ylabel('f(x) [log ölçek]')
axes[1].set_title('Ellipsoid Konverjansı')
axes[1].grid(True, alpha=0.3, which='both')

plt.tight_layout()
plt.savefig('cmaes_ellips.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n📌 Gözlem: Elips başta küresel, sonra ellipsoid'in şekline uyum sağlar!")
print("   Bu adaptasyon gradyan hesaplamadan kovaryansı öğrenmenin gücüdür.")

## 5. Boyut Ölçeklenebilirliği Analizi

CMA-ES'in en güçlü özelliği: Yüksek boyutlarda etkin çalışması.

In [ ]:
# ============================================================
# BOYUT OLÇEKLENEBİLİRLİK ANALİZİ
# ============================================================

dims = [2, 5, 10, 20, 30]    # Test edilecek boyutlar
results_dim = {}

print("Boyut ölçeklenebilirliği analiz ediliyor...")
print(f"{'Boyut':>8} | {'Nesil':>8} | {'f_min':>12} | {'Hata':>12}")
print("-" * 50)

for d in dims:
    np.random.seed(42)
    x0_d = np.random.uniform(-3, 3, d)
    
    best_s, best_f, hist_d = run_cmaes(
        fitness_fn = rosenbrock,
        n_dims     = d,
        sigma0     = 1.0,
        max_gens   = 1000,
        tol        = 1e-6,
        verbose    = False    # Detaylı çıktı kapat
    )
    
    # Rosenbrock minimum = 0 @ (1,...,1)
    error = np.mean(np.abs(best_s - 1.0))
    results_dim[d] = {'best': best_f, 'error': error, 'gens': len(hist_d['best'])}
    print(f"{d:>8} | {len(hist_d['best']):>8} | {best_f:>12.2e} | {error:>12.2e}")


# Görselleştirme
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

dims_plot  = list(results_dim.keys())
errors     = [results_dim[d]['error'] for d in dims_plot]
gen_counts = [results_dim[d]['gens']  for d in dims_plot]

axes[0].semilogy(dims_plot, errors, 'bo-', linewidth=2, markersize=8)
axes[0].set_xlabel('Problem Boyutu (n)')
axes[0].set_ylabel('Ortalama Hata |x - 1|')
axes[0].set_title('Boyuta Göre Optimizasyon Hatası')
axes[0].grid(True, alpha=0.3)

axes[1].plot(dims_plot, gen_counts, 'ro-', linewidth=2, markersize=8)
axes[1].set_xlabel('Problem Boyutu (n)')
axes[1].set_ylabel('Gereken Nesil Sayısı')
axes[1].set_title('Boyuta Göre Hesaplama Maliyeti')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('cmaes_boyut.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n📌 CMA-ES, yüksek boyutlarda gradient tabanlı yöntemlere rakip olabilir!")

## 6. Özet

### CMA-ES'in Gücü
- **Otomatik adaptasyon**: Problem geometrisini kendi kendine öğrenir
- **Rotasyonel değişmezlik**: Koordinat sisteminden bağımsız
- **Ölçek değişmezliği**: Farklı eksen ölçeklerinde çalışır
- **Gradient-free**: Türev hesaplamaya gerek yok

### Ne Zaman Kullanılmalı?
- 10-1000 boyutlu sürekli optimizasyon
- Gradyanın hesaplanamadığı veya güvenilmez olduğu problemler
- Ill-conditioned (kötü koşullu) fonksiyonlar
- Makine öğrenimi hiperparametre optimizasyonu